# Bab 1: Judul & Overview
## ISFEST 2026 Data Competition - Universitas Multimedia Nusantara
### Eksperimen 4: Dual-Philosophy 6-Model Stacking dengan Pembobotan Musim Dingin Terkalibrasi dan Presisi Kontinu Penuh

Notebook ini merupakan berkas eksperimen mandiri resmi ke-4 dari Piji (Tim MAKAN ITU PENTING) pada kompetisi pemodelan ISFEST 2026. Berdasarkan evaluasi submission ke-3 yang berhasil memecahkan rekor validasi lokal terbaik tim (RMSE 0.067678) dan mencetak skor 0.06795 di Kaggle Leaderboard, eksperimen 4 dirancang sebagai terobosan penentu untuk menembus zona papan atas Top Leaderboard (skor 0.06749).

Strategi pemodelan pada Eksperimen 4 mengintegrasikan lima inovasi utama:
1. Presisi Kontinu Penuh: Menghilangkan kuantisasi pembulatan tiga desimal dan mempertahankan format floating-point presisi ganda untuk mengeliminasi variansi kuantisasi acak pada metrik evaluasi RMSE.
2. Kompensasi Pergeseran Rata-rata Musim Dingin (Winter Target Drift): Fitur rasio utilisasi stasiun 28 hari terakhir terhadap rata-rata historis (st_recent28_ratio) serta penalti kuadratik suhu beku (temp_freeze_penalty_sq).
3. Spesialisasi Koridor Jalan Tol dan Jam Sibuk: Profil target makro tiga dimensi pada tipe lokasi, jam diurnal, dan status akhir pekan untuk meredam lonjakan galat di koridor tol.
4. Pembobotan Sampel Terarah (Temporal Sample Weighting): Pemberian bobot pelatihan proporsional pada fase transisi musim dingin bulan November dan jam-jam sibuk siang hari (jam 10:00 hingga 18:00).
5. Arsitektur Dual-Philosophy 6-Model Stacking: Penggabungan terpadu dalam satu notebook antara Aliran A (Deep Capacity Boosting: LightGBM, CatBoost GPU, XGBoost GPU dengan kedalaman 8-10) dan Aliran B (Conservative Regularized Boosting: LightGBM, CatBoost GPU, XGBoost GPU dengan kedalaman 6 dan regularisasi L2 ketat), yang dipadukan menggunakan Non-Negative Ridge Meta-Learner.

Seluruh kode disusun dengan kepatuhan penuh terhadap standar kode bersih (strict clean code), tanpa komentar inline dengan tanda pagar, tanpa simbol dekoratif berlebihan, dan 100% mandiri tanpa ketergantungan berkas eksternal.


# Bab 2: Import Libraries & Setup
Pemasangan pustaka CatBoost dan gdown secara otomatis, konfigurasi visualisasi grafis standar industri data sains, serta inisialisasi akselerasi GPU Tesla P100 / T4.


In [ ]:
!pip install catboost gdown -q


Impor seluruh pustaka komputasi numerik, pemodelan gradient boosting heterogen, evaluasi metrik, dan optimasi matematis.


In [ ]:
import os
import gc
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

gpu_available = False
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_available = True
        print(f"Akselerator GPU Terdeteksi: {gpu_name}")
    else:
        print("Akselerator GPU tidak aktif. Berjalan pada Multi-Threaded CPU.")
except ImportError:
    print("PyTorch tidak terpasang. Berjalan pada Multi-Threaded CPU.")


# Bab 3: Load Data
Pengunduhan dataset secara otomatis dari Google Drive resmi tim melalui pustaka gdown jika berkas belum ada di direktori kerja, serta resolusi jalur berkas secara fleksibel.


In [ ]:
GDRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/16kkQIyF5Yj3y3xIImN9kkewJQ0ZGt_ZH?usp=sharing'

def download_data_from_gdrive():
    target_files = ['train.csv', 'test.csv', 'sample_submission.csv']
    all_exist = all(os.path.exists(f) or os.path.exists(os.path.join('data', f)) for f in target_files)
    if not all_exist:
        print("Mengunduh dataset resmi tim dari Google Drive via gdown...")
        os.system(f'gdown --folder "{GDRIVE_FOLDER_URL}" -O ./data_gdrive --remaining-ok')
        for root, dirs, files in os.walk('.'):
            for f in files:
                if f in target_files and not os.path.exists(f):
                    src = os.path.join(root, f)
                    dst = f
                    try:
                        import shutil
                        shutil.copyfile(src, dst)
                    except Exception:
                        pass

def resolve_data_paths():
    download_data_from_gdrive()
    search_dirs = [
        '.',
        './data',
        '../data',
        './data_gdrive',
        '/kaggle/input/datasets/rabbaniyuki/isfest-dataset',
        '/kaggle/input/isfest-dataset',
        '/kaggle/input/ev-charging-demand-indonesian-student-competition'
    ]
    train_found = None
    test_found = None
    sample_sub_found = None
    for d in search_dirs:
        tr = os.path.join(d, 'train.csv')
        te = os.path.join(d, 'test.csv')
        su = os.path.join(d, 'sample_submission.csv')
        if os.path.exists(tr) and train_found is None:
            train_found = tr
        if os.path.exists(te) and test_found is None:
            test_found = te
        if os.path.exists(su) and sample_sub_found is None:
            sample_sub_found = su
    if train_found is None or test_found is None:
        raise FileNotFoundError("Berkas train.csv atau test.csv tidak ditemukan pada sistem.")
    return train_found, test_found, sample_sub_found

train_file_path, test_file_path, sample_sub_file_path = resolve_data_paths()
print(f"Jalur Data Latih: {train_file_path}")
print(f"Jalur Data Uji  : {test_file_path}")

train_raw = pd.read_csv(train_file_path)
test_raw = pd.read_csv(test_file_path)

print(f"Dimensi Data Latih: {train_raw.shape[0]:,} baris x {train_raw.shape[1]} kolom")
print(f"Dimensi Data Uji  : {test_raw.shape[0]:,} baris x {test_raw.shape[1]} kolom")


Optimasi efisiensi alokasi memori RAM melalui konversi presisi tipe data numerik.


In [ ]:
def optimize_memory(df):
    initial_mem = df.memory_usage().sum() / (1024**2)
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not str(col_type).startswith('datetime'):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type).startswith('int'):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            elif str(col_type).startswith('float'):
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    final_mem = df.memory_usage().sum() / (1024**2)
    print(f"Efisiensi Memori: {initial_mem:.1f} MB -> {final_mem:.1f} MB (Hemat {100*(initial_mem-final_mem)/initial_mem:.1f}%)")
    return df

train_raw = optimize_memory(train_raw)
test_raw = optimize_memory(test_raw)


# Bab 4: Exploratory Data Analysis (EDA)
Eksplorasi data mendalam untuk mengidentifikasi karakteristik distribusi variabel target, pola temporal musiman, konsentrasi galat jam sibuk, anomali koridor tol, dan korelasi antar fitur numerik.


### 4.1 Struktur Data dan Informasi Tipe Kolom
Pemeriksaan ringkasan informasi tipe data dan jumlah baris terisi pada dataset pelatihan.


In [ ]:
train_raw.info()


### 4.2 Analisis Sebaran Nilai Kosong (Missing Values)
Pengecekan kuantitas dan persentase nilai kosong pada data latih dan data uji.


In [ ]:
missing_tr = train_raw.isnull().sum()
missing_te = test_raw.isnull().sum()

missing_report = pd.DataFrame({
    'Train Missing': missing_tr[missing_tr > 0],
    'Train Pct (%)': (missing_tr[missing_tr > 0] / len(train_raw) * 100).round(2),
    'Test Missing': missing_te[missing_te > 0],
    'Test Pct (%)': (missing_te[missing_te > 0] / len(test_raw) * 100).round(2)
})
print(missing_report)


### 4.3 Sebaran Statistik Variabel Target (utilization_rate)
Visualisasi kurva distribusi probabilitas dan deteksi sebaran statistik variabel target kontinua.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(train_raw['utilization_rate'], bins=60, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Distribusi Variabel Target (utilization_rate)')
axes[0].set_xlabel('Tingkat Utilisasi')
axes[0].set_ylabel('Frekuensi')

sns.boxplot(x=train_raw['utilization_rate'], ax=axes[1], color='#4a90e2')
axes[1].set_title('Deteksi Pencilan Statistik Target')
axes[1].set_xlabel('Tingkat Utilisasi')

plt.tight_layout()
plt.show()

print("Statistik Deskriptif Target:")
print(train_raw['utilization_rate'].describe().round(4))


### 4.4 Kurva Fluktuasi Diurnal Jam Sibuk (Hari Kerja vs Akhir Pekan)
Analisis tingkat utilisasi sepanjang 24 jam dengan pemisahan karakteristik hari kerja dan akhir pekan.


In [ ]:
temp_eda_dt = pd.to_datetime(train_raw['timestamp'], format='mixed')
train_raw_eda = train_raw.copy()
train_raw_eda['hour'] = temp_eda_dt.dt.hour
train_raw_eda['is_weekend'] = temp_eda_dt.dt.dayofweek.isin([5, 6]).map({True: 'Akhir Pekan', False: 'Hari Kerja'})

hourly_pattern = train_raw_eda.groupby(['hour', 'is_weekend'])['utilization_rate'].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=hourly_pattern, x='hour', y='utilization_rate', hue='is_weekend', marker='o', palette=['#1f77b4', '#ff7f0e'])
plt.title('Kurva Fluktuasi Beban Utilisasi Diurnal (24 Jam)')
plt.xlabel('Jam (0 - 23)')
plt.ylabel('Rata-rata Utilisasi')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### 4.5 Profil Beban Utilisasi Berdasarkan Tipe Lokasi dan Jenis Konektor Charger
Perbandingan rata-rata keterisian fasilitas di berbagai simpul mobilitas publik serta klasifikasi teknologi konektor daya.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

loc_order = train_raw.groupby('location_type')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='utilization_rate', y='location_type', order=loc_order, ax=axes[0], palette='Blues_r', ci=None)
axes[0].set_title('Rata-rata Utilisasi Berdasarkan Tipe Lokasi Fasilitas')
axes[0].set_xlabel('Rata-rata Utilisasi')
axes[0].set_ylabel('Tipe Lokasi')

chg_order = train_raw.groupby('charger_type')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='utilization_rate', y='charger_type', order=chg_order, ax=axes[1], palette='crest', ci=None)
axes[1].set_title('Rata-rata Utilisasi Berdasarkan Jenis Konektor Charger')
axes[1].set_xlabel('Rata-rata Utilisasi')
axes[1].set_ylabel('Jenis Charger')

plt.tight_layout()
plt.show()


### 4.6 Pengaruh Kondisi Cuaca Ekstrem dan Suhu Lingkungan terhadap Utilisasi
Eksplorasi hubungan termodinamika suhu lingkungan terhadap laju pemanfaatan stasiun pengisian daya EV.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

weather_order = train_raw.groupby('weather_condition')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='utilization_rate', y='weather_condition', order=weather_order, ax=axes[0], palette='mako', ci=None)
axes[0].set_title('Rata-rata Utilisasi Berdasarkan Kondisi Cuaca')
axes[0].set_xlabel('Rata-rata Utilisasi')
axes[0].set_ylabel('Kondisi Cuaca')

temp_binned = pd.cut(train_raw['temperature_f'], bins=10)
temp_util = train_raw.groupby(temp_binned)['utilization_rate'].mean()
temp_util.plot(kind='bar', ax=axes[1], color='#34495e', rot=45)
axes[1].set_title('Rata-rata Utilisasi Berdasarkan Rentang Suhu (Fahrenheit)')
axes[1].set_xlabel('Rentang Suhu')
axes[1].set_ylabel('Rata-rata Utilisasi')

plt.tight_layout()
plt.show()


### 4.7 Matriks Korelasi Linear Antar Variabel Numerik
Perhitungan koefisien korelasi Pearson antar fitur kontinu terhadap variabel target pemanfaatan stasiun.


In [ ]:
num_cols_eda = train_raw.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = train_raw[num_cols_eda].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='vlag', vmin=-1.0, vmax=1.0, linewidths=0.5)
plt.title('Matriks Korelasi Pearson Fitur Numerik')
plt.tight_layout()
plt.show()


### 4.8 Audit Tiga Anomali Spesifik Sesuai Panduan Panitia ISFEST 2026
Audit identitas ganda pada stasiun dengan nama identik serta kontinuitas baris data pengujian pada 31 Desember 2025.


In [ ]:
st_name_audit = train_raw.groupby('station_name')['station_id'].nunique()
dup_st_names = st_name_audit[st_name_audit > 1]
print("Audit Stasiun dengan Nama Serupa Namun ID Berbeda:")
for name, cnt in dup_st_names.items():
    ids = train_raw[train_raw['station_name'] == name]['station_id'].unique().tolist()
    print(f"  Stasiun '{name}' memiliki {cnt} ID berbeda: {ids}")
print("Keputusan Desain: station_id ditetapkan sebagai entitas spasial primer pemodelan.")

dec31_data = test_raw[pd.to_datetime(test_raw['timestamp'], format='mixed').dt.date == pd.to_datetime('2025-12-31').date()]
print(f"Jumlah Baris Pengujian pada 31 Desember 2025: {len(dec31_data)} baris")
print(f"Jam Tercatat pada 31 Desember 2025: {pd.to_datetime(dec31_data['timestamp'], format='mixed').dt.hour.unique().tolist()}")


# Bab 5: Data Cleaning
Pembersihan data menerapkan imputasi temporal terarah forward fill dan backward fill per stasiun, dilanjutkan dengan pengisian median per kota dan jam. Struktur baris dipertahankan secara murni tanpa mengubah urutan indeks asli dataset.


In [ ]:
def clean_and_impute_dataset(df_in):
    df = df_in.copy()
    df['datetime'] = pd.to_datetime(df['timestamp'], format='mixed')
    
    df['temperature_f'] = df.groupby('station_id')['temperature_f'].ffill().bfill()
    df['precipitation_mm'] = df.groupby('station_id')['precipitation_mm'].ffill().bfill()
    
    city_hr_temp = df.groupby(['city', df['datetime'].dt.hour])['temperature_f'].transform(lambda s: s.fillna(s.median()))
    df['temperature_f'] = df['temperature_f'].fillna(city_hr_temp).fillna(df['temperature_f'].median())
    
    df['precipitation_mm'] = df['precipitation_mm'].fillna(0.0)
    
    df['weather_condition'] = df['weather_condition'].fillna('clear')
    df['local_event'] = df['local_event'].fillna('none')
    df['pricing_type'] = df['pricing_type'].fillna('per_kwh')
    
    return df

print("Menjalankan pembersihan dan imputasi data latih dan data uji...")
train_clean = clean_and_impute_dataset(train_raw)
test_clean = clean_and_impute_dataset(test_raw)

print(f"Nilai kosong pada data latih setelah imputasi: {train_clean.isnull().sum().sum()}")
print(f"Nilai kosong pada data uji setelah imputasi  : {test_clean.isnull().sum().sum()}")


# Bab 6: Feature Engineering
Pembangunan fitur prediktif mutakhir mencakup:
1. Waktu granular kontinu dan transformasi siklikal trigonometri resolusi 48 interval.
2. Penanda kalender kuartal empat dan pekan libur akhir tahun (Thanksgiving, Natal, Malam Tahun Baru).
3. Termodinamika baterai non-linear pada cuaca dingin ekstrem (battery cold penalty kuadratik) serta deviasi iklim mikro lokal kota.
4. Spesifikasi daya port, kapasitas total stasiun, dan rasio disparitas harga bahan bakar minyak terhadap rata-rata kota.
5. Interaksi spasial jam sibuk pada klaster perkantoran, pusat belanja, dan koridor jalan tol.
6. Multi-hot parsing fasilitas sekitar (amenities).
7. Profil target makro Bayesian m-estimate (m=15) 5 pilar teruji, rasio utilisasi 28 hari terakhir, serta profil spasial koridor jalan tol per jam dan hari libur.


In [ ]:
def engineer_base_features(df):
    out = df.copy()
    if 'datetime' not in out.columns:
        out['datetime'] = pd.to_datetime(out['timestamp'], format='mixed')
        
    out['hour'] = out['datetime'].dt.hour
    out['minute'] = out['datetime'].dt.minute
    out['time_float'] = (out['hour'] + out['minute'] / 60.0).astype(np.float32)
    out['dayofweek'] = out['datetime'].dt.dayofweek
    out['day'] = out['datetime'].dt.day
    out['month'] = out['datetime'].dt.month
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    out['weekofyear'] = out['datetime'].dt.isocalendar().week.astype(int)
    
    out['sin_hour'] = np.sin(2 * np.pi * out['time_float'] / 24.0).astype(np.float32)
    out['cos_hour'] = np.cos(2 * np.pi * out['time_float'] / 24.0).astype(np.float32)
    out['sin_dow'] = np.sin(2 * np.pi * out['dayofweek'] / 7.0).astype(np.float32)
    out['cos_dow'] = np.cos(2 * np.pi * out['dayofweek'] / 7.0).astype(np.float32)
    
    out['is_thanksgiving_week'] = ((out['month'] == 11) & (out['day'] >= 24) & (out['day'] <= 30)).astype(int)
    out['is_christmas_week'] = ((out['month'] == 12) & (out['day'] >= 20) & (out['day'] <= 26)).astype(int)
    out['is_nye'] = ((out['month'] == 12) & (out['day'] >= 29)).astype(int)
    
    out['is_freezing'] = ((out['temperature_f'] <= 32.0) | (out['weather_condition'] == 'freezing')).astype(int)
    out['battery_cold_penalty'] = np.maximum(0.0, 32.0 - out['temperature_f']).astype(np.float32)
    out['battery_cold_penalty_sq'] = (out['battery_cold_penalty'] ** 2).astype(np.float32)
    out['is_extreme_heat'] = ((out['temperature_f'] >= 95.0) | (out['weather_condition'] == 'extreme_heat')).astype(int)
    out['is_raining'] = (out['precipitation_mm'] > 0.0).astype(int)
    
    city_hr_temp = out.groupby(['city', 'hour'])['temperature_f'].transform('mean')
    out['temp_dev_city_hour'] = (out['temperature_f'] - city_hr_temp).astype(np.float32)
    
    city_gas_avg = out.groupby('city')['gas_price_per_gallon'].transform('mean')
    out['gas_price_ratio_city'] = (out['gas_price_per_gallon'] / city_gas_avg.replace(0, 1.0)).astype(np.float32)
    out['gas_price_per_kw'] = (out['gas_price_per_gallon'] / (out['power_output_kw'] / 50.0).replace(0, 1.0)).astype(np.float32)
    
    out['ports_total_safe'] = out['ports_total'].replace(0, 1)
    out['power_per_port'] = (out['power_output_kw'] / out['ports_total_safe']).astype(np.float32)
    out['station_total_capacity_kw'] = (out['power_output_kw'] * out['ports_total']).astype(np.float32)
    out['is_ultra_fast'] = (out['power_output_kw'] >= 150.0).astype(int)
    out['is_free_pricing'] = (out['pricing_type'].astype(str).str.lower() == 'free').astype(int)
    
    out['is_workplace_peak'] = ((out['location_type'] == 'Workplace') & (out['is_weekend'] == 0) & (out['hour'].between(8, 17))).astype(int)
    out['is_mall_peak'] = ((out['location_type'] == 'Shopping Mall') & (out['hour'].between(12, 20))).astype(int)
    out['is_highway_peak'] = ((out['location_type'] == 'Highway Corridor') & (out['hour'].between(10, 19))).astype(int)
    out['is_residential_night'] = ((out['location_type'] == 'Residential') & ((out['hour'] >= 20) | (out['hour'] <= 6))).astype(int)
    out['freezing_highway'] = (out['is_freezing'] * out['is_highway_peak']).astype(int)
    
    out['has_local_event'] = (out['local_event'].fillna('none').astype(str).str.lower() != 'none').astype(int)
    
    amenities_list = ['WiFi', 'Restroom', 'Shopping Mall', 'Park', 'Fast Food', 'Hotel', 'Convenience Store', 'Grocery Store']
    for amen in amenities_list:
        col_name = 'has_' + amen.lower().replace(' ', '_')
        out[col_name] = out['amenities_nearby'].fillna('').astype(str).str.contains(amen, case=False, regex=False).astype(int)
    out['total_amenities_count'] = out[[c for c in out.columns if c.startswith('has_') and c != 'has_local_event']].sum(axis=1)
    
    return out

train_base = engineer_base_features(train_clean)
test_base = engineer_base_features(test_clean)
print(f"Dimensi fitur dasar data latih: {train_base.shape}")
print(f"Dimensi fitur dasar data uji  : {test_base.shape}")


### 6.2 Hierarchical Bayesian Target Profiles dan Rasio Transisi Musiman
Perhitungan target encoding bebas kebocoran data dengan bobot m-estimate 15 pada hierarki stasiun, jam, tipe lokasi, operator jaringan, serta rasio dinamika 28 hari terakhir terhadap baseline stasiun.


In [ ]:
TARGET_PROFILE_COLS = [
    'target_prof_st_hr_wk',
    'target_prof_st_hr',
    'target_prof_st',
    'target_prof_loc_hr',
    'target_prof_net_hr',
    'target_prof_loc_hr_wk',
    'target_prof_st_recent28',
    'st_recent28_ratio'
]

def compute_hierarchical_target_profiles(train_source, *target_dfs, m_weight=15.0):
    global_mean = train_source['utilization_rate'].mean()

    def smooth_agg(group_keys, col_name):
        agg_df = train_source.groupby(group_keys, observed=False)['utilization_rate'].agg(['count', 'mean']).reset_index()
        agg_df[col_name] = (agg_df['count'] * agg_df['mean'] + m_weight * global_mean) / (agg_df['count'] + m_weight)
        return agg_df[group_keys + [col_name]]

    st_hr_wk_prof = smooth_agg(['station_id', 'hour', 'is_weekend'], 'target_prof_st_hr_wk')
    st_hr_prof = smooth_agg(['station_id', 'hour'], 'target_prof_st_hr')
    st_prof = smooth_agg(['station_id'], 'target_prof_st')
    loc_hr_prof = smooth_agg(['location_type', 'hour'], 'target_prof_loc_hr')
    net_hr_prof = smooth_agg(['network', 'hour'], 'target_prof_net_hr')
    loc_hr_wk_prof = smooth_agg(['location_type', 'hour', 'is_weekend'], 'target_prof_loc_hr_wk')
    
    max_train_date = train_source['datetime'].max()
    recent_cutoff = max_train_date - pd.Timedelta(days=28)
    recent_source = train_source[train_source['datetime'] > recent_cutoff]
    
    agg_recent = recent_source.groupby('station_id', observed=False)['utilization_rate'].agg(['count', 'mean']).reset_index()
    agg_recent['target_prof_st_recent28'] = (agg_recent['count'] * agg_recent['mean'] + m_weight * global_mean) / (agg_recent['count'] + m_weight)
    recent_st_prof = agg_recent[['station_id', 'target_prof_st_recent28']]

    def merge_profiles(df):
        out = df.copy()
        existing = [c for c in TARGET_PROFILE_COLS if c in out.columns]
        if len(existing) > 0:
            out = out.drop(columns=existing)

        out = out.merge(st_hr_wk_prof, on=['station_id', 'hour', 'is_weekend'], how='left')
        out = out.merge(st_hr_prof, on=['station_id', 'hour'], how='left')
        out = out.merge(st_prof, on=['station_id'], how='left')
        out = out.merge(loc_hr_prof, on=['location_type', 'hour'], how='left')
        out = out.merge(net_hr_prof, on=['network', 'hour'], how='left')
        out = out.merge(loc_hr_wk_prof, on=['location_type', 'hour', 'is_weekend'], how='left')
        out = out.merge(recent_st_prof, on=['station_id'], how='left')

        out['target_prof_st_hr_wk'] = out['target_prof_st_hr_wk'].fillna(out['target_prof_st_hr']).fillna(out['target_prof_st']).fillna(global_mean)
        out['target_prof_st_hr'] = out['target_prof_st_hr'].fillna(out['target_prof_st']).fillna(global_mean)
        out['target_prof_st'] = out['target_prof_st'].fillna(global_mean)
        out['target_prof_loc_hr'] = out['target_prof_loc_hr'].fillna(global_mean)
        out['target_prof_net_hr'] = out['target_prof_net_hr'].fillna(global_mean)
        out['target_prof_loc_hr_wk'] = out['target_prof_loc_hr_wk'].fillna(out['target_prof_loc_hr']).fillna(global_mean)
        out['target_prof_st_recent28'] = out['target_prof_st_recent28'].fillna(out['target_prof_st']).fillna(global_mean)
        out['st_recent28_ratio'] = (out['target_prof_st_recent28'] / out['target_prof_st'].replace(0, global_mean)).astype(np.float32)
        return out

    transformed = [merge_profiles(train_source)]
    for target_df in target_dfs:
        transformed.append(merge_profiles(target_df))
    return transformed if len(transformed) > 1 else transformed[0]

print("Menghitung Hierarchical Bayesian Target Profiles dan Rasio Dinamika Musim Dingin...")
train_feat, test_feat = compute_hierarchical_target_profiles(train_base, test_base)
print("Penggabungan 8 Fitur Profil Target Hirarkis Berhasil.")


# Bab 7: Feature Selection
Penyusunan matriks fitur akhir dan enkoding kolom kategori untuk akselerasi pelatihan model berbasis pohon keputusan.


In [ ]:
DROP_COLS = [
    'id', 'timestamp', 'datetime', 'station_name', 'amenities_nearby',
    'utilization_rate', 'ports_total_safe'
]

FEATURE_COLS = [c for c in train_feat.columns if c not in DROP_COLS]

CATEGORICAL_COLS = [
    'station_id', 'network', 'city', 'state', 'location_type',
    'charger_type', 'pricing_type', 'weather_condition', 'local_event'
]

for c in CATEGORICAL_COLS:
    train_feat[c] = train_feat[c].fillna('missing').astype('category')
    test_feat[c] = test_feat[c].fillna('missing').astype('category')

X_train_all = train_feat[FEATURE_COLS]
y_train_all = train_feat['utilization_rate'].values
X_test_all = test_feat[FEATURE_COLS]

print(f"Jumlah Fitur Final Terpilih: {len(FEATURE_COLS)}")
print(f"Dimensi Matriks Fitur Latih Penuh : {X_train_all.shape}")
print(f"Dimensi Matriks Fitur Uji Penuh   : {X_test_all.shape}")


# Bab 8: Train-Test-Validation Split & Cross-Validation Strategy
Pemisahan partisi validasi out-of-time pada 7 hari terakhir data latih untuk simulasi pengujian temporal bebas kebocoran data. Pada bab ini juga dihitung bobot sampel temporal (sample weights) yang memberikan fokus lebih tinggi pada data bulan November dan jam-jam sibuk siang hari.


In [ ]:
train_sorted = train_base.sort_values('datetime').reset_index(drop=True)
val_cutoff_time = train_sorted['datetime'].max() - pd.Timedelta(days=7)

tr_mask = train_sorted['datetime'] <= val_cutoff_time
va_mask = train_sorted['datetime'] > val_cutoff_time

raw_tr_part = train_sorted.loc[tr_mask].copy()
raw_va_part = train_sorted.loc[va_mask].copy()

tr_part, va_part = compute_hierarchical_target_profiles(raw_tr_part, raw_va_part)

for c in CATEGORICAL_COLS:
    tr_part[c] = tr_part[c].astype('category')
    va_part[c] = va_part[c].astype('category')

X_tr = tr_part[FEATURE_COLS].copy()
y_tr = tr_part['utilization_rate'].values
X_va = va_part[FEATURE_COLS].copy()
y_va = va_part['utilization_rate'].values

weights_tr = np.ones(len(tr_part), dtype=np.float32)
nov_mask = (tr_part['month'] == 11).values
peak_mask = (tr_part['hour'].between(10, 18)).values
weights_tr[nov_mask] *= 1.15
weights_tr[nov_mask & peak_mask] *= 1.10

weights_full = np.ones(len(train_feat), dtype=np.float32)
nov_full_mask = (train_feat['month'] == 11).values
peak_full_mask = (train_feat['hour'].between(10, 18)).values
weights_full[nov_full_mask] *= 1.15
weights_full[nov_full_mask & peak_full_mask] *= 1.10

print(f"Batas Waktu Validasi Out-of-Time : {val_cutoff_time}")
print(f"Jumlah Baris Latih Partisi       : {len(X_tr):,} baris")
print(f"Jumlah Baris Validasi Partisi    : {len(X_va):,} baris")
print(f"Rentang Bobot Sampel Pelatihan   : [{weights_tr.min():.2f}, {weights_tr.max():.2f}]")


# Bab 9: Baseline Model
Model regresi linear Ridge sebagai tolok ukur dasar kinerja prediktif pada fitur numerik.


In [ ]:
num_cols_only = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_tr[num_cols_only].fillna(0), y_tr)

preds_val_base = np.clip(baseline_model.predict(X_va[num_cols_only].fillna(0)), 0.02, 0.98)

rmse_base = root_mean_squared_error(y_va, preds_val_base)
mae_base = mean_absolute_error(y_va, preds_val_base)
r2_base = r2_score(y_va, preds_val_base)

print(f"Evaluasi Model Patokan Dasar (Baseline Ridge):")
print(f"  Root Mean Squared Error (RMSE) : {rmse_base:.5f}")
print(f"  Mean Absolute Error (MAE)      : {mae_base:.5f}")
print(f"  Koefisien Determinasi (R2)     : {r2_base:.5f}")


# Bab 10: Modeling
Arsitektur pemodelan Dual-Philosophy 6-Model Stacking menggabungkan dua filosofi boosting komplementer:
- Aliran A (Deep Capacity Boosting): LightGBM Deep (depth 10, leaves 127), CatBoost Deep GPU (depth 8), dan XGBoost Deep GPU (depth 8) untuk menangkap interaksi spasial-temporal kompleks.
- Aliran B (Conservative Regularized Boosting): LightGBM Reg (depth 6, leaves 31, reg L2 tinggi), CatBoost Reg GPU (depth 6, L2 reg 6.0), dan XGBoost Reg GPU (depth 6, subsample 0.70) untuk meredam variansi pada jam puncak.

Setiap model dievaluasi pada partisi validasi out-of-time untuk menentukan jumlah iterasi konvergen optimal sebelum dilakukan pelatihan penuh.


In [ ]:
SEEDS = [42, 100, 2024]

lgb_A_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'max_depth': 10,
    'learning_rate': 0.05,
    'n_estimators': 1500,
    'subsample': 0.85,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'verbose': -1
}

cb_A_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 1500,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'verbose': 0
}
if gpu_available:
    cb_A_params['task_type'] = 'GPU'
else:
    cb_A_params['thread_count'] = -1

xgb_A_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 1500,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.85,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 40
}
if gpu_available:
    xgb_A_params['device'] = 'cuda'

lgb_B_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'max_depth': 6,
    'learning_rate': 0.04,
    'n_estimators': 1500,
    'subsample': 0.70,
    'colsample_bytree': 0.70,
    'reg_alpha': 0.5,
    'reg_lambda': 3.0,
    'n_jobs': -1,
    'verbose': -1
}

cb_B_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 1500,
    'learning_rate': 0.04,
    'depth': 6,
    'l2_leaf_reg': 6.0,
    'verbose': 0
}
if gpu_available:
    cb_B_params['task_type'] = 'GPU'
else:
    cb_B_params['thread_count'] = -1

xgb_B_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 1500,
    'learning_rate': 0.04,
    'max_depth': 6,
    'subsample': 0.70,
    'colsample_bytree': 0.70,
    'reg_alpha': 0.5,
    'reg_lambda': 3.0,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 40
}
if gpu_available:
    xgb_B_params['device'] = 'cuda'

print("Konfigurasi Parameter Dual-Philosophy 6-Model Berhasil Diinisialisasi.")


Pelatihan Model Aliran A: Deep Capacity Boosting (A1 LightGBM, A2 CatBoost GPU, A3 XGBoost GPU) pada partisi validasi out-of-time.


In [ ]:
print("Melatih Model A1: LightGBM Deep pada Partisi Validasi...")
lgb_A_val_preds = []
best_iters_lgb_A = []
for s in SEEDS:
    m = lgb.LGBMRegressor(**{**lgb_A_params, 'random_state': s})
    m.fit(
        X_tr, y_tr,
        sample_weight=weights_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    best_iters_lgb_A.append(m.best_iteration_)
    lgb_A_val_preds.append(np.clip(m.predict(X_va), 0.02, 0.98))
val_pred_lgb_A = np.mean(lgb_A_val_preds, axis=0)
optimal_lgb_A_iter = int(np.median(best_iters_lgb_A))
print(f"Model A1 (LightGBM Deep) Validation RMSE : {root_mean_squared_error(y_va, val_pred_lgb_A):.5f} ({optimal_lgb_A_iter} pohon)")

print("Melatih Model A2: CatBoost GPU Deep pada Partisi Validasi...")
X_tr_cb = X_tr.copy()
X_va_cb = X_va.copy()
for cat in CATEGORICAL_COLS:
    X_tr_cb[cat] = X_tr_cb[cat].astype(str)
    X_va_cb[cat] = X_va_cb[cat].astype(str)

cb_A_val_preds = []
best_iters_cb_A = []
for s in SEEDS:
    m = cb.CatBoostRegressor(**{**cb_A_params, 'random_seed': s})
    m.fit(
        X_tr_cb, y_tr,
        sample_weight=weights_tr,
        eval_set=(X_va_cb, y_va),
        cat_features=CATEGORICAL_COLS,
        early_stopping_rounds=40
    )
    best_iters_cb_A.append(m.get_best_iteration())
    cb_A_val_preds.append(np.clip(m.predict(X_va_cb), 0.02, 0.98))
val_pred_cb_A = np.mean(cb_A_val_preds, axis=0)
optimal_cb_A_iter = int(np.median(best_iters_cb_A))
print(f"Model A2 (CatBoost GPU Deep) Validation RMSE : {root_mean_squared_error(y_va, val_pred_cb_A):.5f} ({optimal_cb_A_iter} pohon)")

print("Melatih Model A3: XGBoost GPU Deep pada Partisi Validasi...")
xgb_tr = X_tr.copy()
xgb_va = X_va.copy()
for c in CATEGORICAL_COLS:
    xgb_tr[c] = xgb_tr[c].cat.codes
    xgb_va[c] = xgb_va[c].cat.codes

xgb_A_val_preds = []
best_iters_xgb_A = []
for s in SEEDS:
    m = xgb.XGBRegressor(**{**xgb_A_params, 'random_state': s})
    m.fit(
        xgb_tr, y_tr,
        sample_weight=weights_tr,
        eval_set=[(xgb_va, y_va)],
        verbose=False
    )
    best_iters_xgb_A.append(m.best_iteration)
    xgb_A_val_preds.append(np.clip(m.predict(xgb_va), 0.02, 0.98))
val_pred_xgb_A = np.mean(xgb_A_val_preds, axis=0)
optimal_xgb_A_iter = int(np.median(best_iters_xgb_A))
print(f"Model A3 (XGBoost GPU Deep) Validation RMSE  : {root_mean_squared_error(y_va, val_pred_xgb_A):.5f} ({optimal_xgb_A_iter} pohon)")


Pelatihan Model Aliran B: Conservative Regularized Boosting (B1 LightGBM, B2 CatBoost GPU, B3 XGBoost GPU) pada partisi validasi out-of-time.


In [ ]:
print("Melatih Model B1: LightGBM Regularized pada Partisi Validasi...")
lgb_B_val_preds = []
best_iters_lgb_B = []
for s in SEEDS:
    m = lgb.LGBMRegressor(**{**lgb_B_params, 'random_state': s})
    m.fit(
        X_tr, y_tr,
        sample_weight=weights_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    best_iters_lgb_B.append(m.best_iteration_)
    lgb_B_val_preds.append(np.clip(m.predict(X_va), 0.02, 0.98))
val_pred_lgb_B = np.mean(lgb_B_val_preds, axis=0)
optimal_lgb_B_iter = int(np.median(best_iters_lgb_B))
print(f"Model B1 (LightGBM Reg) Validation RMSE : {root_mean_squared_error(y_va, val_pred_lgb_B):.5f} ({optimal_lgb_B_iter} pohon)")

print("Melatih Model B2: CatBoost GPU Regularized pada Partisi Validasi...")
cb_B_val_preds = []
best_iters_cb_B = []
for s in SEEDS:
    m = cb.CatBoostRegressor(**{**cb_B_params, 'random_seed': s})
    m.fit(
        X_tr_cb, y_tr,
        sample_weight=weights_tr,
        eval_set=(X_va_cb, y_va),
        cat_features=CATEGORICAL_COLS,
        early_stopping_rounds=40
    )
    best_iters_cb_B.append(m.get_best_iteration())
    cb_B_val_preds.append(np.clip(m.predict(X_va_cb), 0.02, 0.98))
val_pred_cb_B = np.mean(cb_B_val_preds, axis=0)
optimal_cb_B_iter = int(np.median(best_iters_cb_B))
print(f"Model B2 (CatBoost GPU Reg) Validation RMSE : {root_mean_squared_error(y_va, val_pred_cb_B):.5f} ({optimal_cb_B_iter} pohon)")

print("Melatih Model B3: XGBoost GPU Regularized pada Partisi Validasi...")
xgb_B_val_preds = []
best_iters_xgb_B = []
for s in SEEDS:
    m = xgb.XGBRegressor(**{**xgb_B_params, 'random_state': s})
    m.fit(
        xgb_tr, y_tr,
        sample_weight=weights_tr,
        eval_set=[(xgb_va, y_va)],
        verbose=False
    )
    best_iters_xgb_B.append(m.best_iteration)
    xgb_B_val_preds.append(np.clip(m.predict(xgb_va), 0.02, 0.98))
val_pred_xgb_B = np.mean(xgb_B_val_preds, axis=0)
optimal_xgb_B_iter = int(np.median(best_iters_xgb_B))
print(f"Model B3 (XGBoost GPU Reg) Validation RMSE  : {root_mean_squared_error(y_va, val_pred_xgb_B):.5f} ({optimal_xgb_B_iter} pohon)")


# Bab 11: Hyperparameter Tuning
Ringkasan perbandingan konfigurasi hyperparameter antara Aliran A (Deep Capacity) dan Aliran B (Conservative Regularized) yang dioptimasi untuk mencegah overfitting.


In [ ]:
tuning_matrix = pd.DataFrame({
    'Aliran & Model': [
        'Stream A1: LightGBM Deep', 'Stream A2: CatBoost GPU Deep', 'Stream A3: XGBoost GPU Deep',
        'Stream B1: LightGBM Reg', 'Stream B2: CatBoost GPU Reg', 'Stream B3: XGBoost GPU Reg'
    ],
    'Kedalaman (Max Depth)': [
        lgb_A_params['max_depth'], cb_A_params['depth'], xgb_A_params['max_depth'],
        lgb_B_params['max_depth'], cb_B_params['depth'], xgb_B_params['max_depth']
    ],
    'Learning Rate': [
        lgb_A_params['learning_rate'], cb_A_params['learning_rate'], xgb_A_params['learning_rate'],
        lgb_B_params['learning_rate'], cb_B_params['learning_rate'], xgb_B_params['learning_rate']
    ],
    'Regularisasi L2': [
        lgb_A_params['reg_lambda'], cb_A_params['l2_leaf_reg'], xgb_A_params['reg_lambda'],
        lgb_B_params['reg_lambda'], cb_B_params['l2_leaf_reg'], xgb_B_params['reg_lambda']
    ],
    'Pohon Optimal Terkalibrasi': [
        optimal_lgb_A_iter, optimal_cb_A_iter, optimal_xgb_A_iter,
        optimal_lgb_B_iter, optimal_cb_B_iter, optimal_xgb_B_iter
    ]
})
print(tuning_matrix.to_string(index=False))


# Bab 12: Model Evaluation
Evaluasi perbandingan metrik kinerja out-of-time seluruh 6 model individual terhadap model dasar patokan baseline, serta inspeksi fitur dengan kontribusi kepentingan tertinggi.


In [ ]:
all_models_eval = pd.DataFrame({
    'Model': [
        'Baseline Ridge',
        'A1: LightGBM Deep', 'A2: CatBoost GPU Deep', 'A3: XGBoost GPU Deep',
        'B1: LightGBM Reg', 'B2: CatBoost GPU Reg', 'B3: XGBoost GPU Reg'
    ],
    'RMSE': [
        rmse_base,
        root_mean_squared_error(y_va, val_pred_lgb_A),
        root_mean_squared_error(y_va, val_pred_cb_A),
        root_mean_squared_error(y_va, val_pred_xgb_A),
        root_mean_squared_error(y_va, val_pred_lgb_B),
        root_mean_squared_error(y_va, val_pred_cb_B),
        root_mean_squared_error(y_va, val_pred_xgb_B)
    ],
    'MAE': [
        mae_base,
        mean_absolute_error(y_va, val_pred_lgb_A),
        mean_absolute_error(y_va, val_pred_cb_A),
        mean_absolute_error(y_va, val_pred_xgb_A),
        mean_absolute_error(y_va, val_pred_lgb_B),
        mean_absolute_error(y_va, val_pred_cb_B),
        mean_absolute_error(y_va, val_pred_xgb_B)
    ],
    'R2': [
        r2_base,
        r2_score(y_va, val_pred_lgb_A),
        r2_score(y_va, val_pred_cb_A),
        r2_score(y_va, val_pred_xgb_A),
        r2_score(y_va, val_pred_lgb_B),
        r2_score(y_va, val_pred_cb_B),
        r2_score(y_va, val_pred_xgb_B)
    ]
}).sort_values('RMSE')

print("Tabel Perbandingan Kinerja Validasi Out-of-Time:")
print(all_models_eval.to_string(index=False))


Visualisasi 15 fitur dengan tingkat kepentingan tertinggi pada model ensemble terpilih.


In [ ]:
feat_imp_series = pd.DataFrame({
    'Fitur': FEATURE_COLS,
    'Importance': m.feature_importances_ if hasattr(m, 'feature_importances_') else 0
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_series.head(15), x='Importance', y='Fitur', palette='viridis')
plt.title('15 Fitur Paling Berpengaruh pada Model Prediktif')
plt.xlabel('Tingkat Kepentingan Fitur')
plt.ylabel('Nama Fitur')
plt.tight_layout()
plt.show()


# Bab 13: Ensembling & Stacking (Meta-Learning)
Penggabungan terpadu ke-6 model heterogen menggunakan Non-Negative Ridge Stacking Meta-Learner dengan regularisasi L2 (alpha=15.0). Seluruh bobot dihitung secara objektif berdasarkan matriks prediksi validasi out-of-time untuk mencegah bobot negatif yang rentan overfit.


In [ ]:
print("Mengoptimasi Meta-Learner Stacking 6-Model Dual-Philosophy...")
S_val_matrix = np.column_stack([
    val_pred_lgb_A, val_pred_cb_A, val_pred_xgb_A,
    val_pred_lgb_B, val_pred_cb_B, val_pred_xgb_B
])

meta_learner = Ridge(alpha=15.0, positive=True, fit_intercept=False)
meta_learner.fit(S_val_matrix, y_va)

stacking_val_preds = np.clip(meta_learner.predict(S_val_matrix), 0.02, 0.98)
rmse_stacked_val = root_mean_squared_error(y_va, stacking_val_preds)

model_labels = [
    'Stream A1: LightGBM Deep', 'Stream A2: CatBoost GPU Deep', 'Stream A3: XGBoost GPU Deep',
    'Stream B1: LightGBM Reg', 'Stream B2: CatBoost GPU Reg', 'Stream B3: XGBoost GPU Reg'
]

print("Bobot Meta-Learner Terkalibrasi:")
for lbl, w in zip(model_labels, meta_learner.coef_):
    print(f"  {lbl:30s} : {w:.4f}")
print(f"Validasi RMSE Hasil Stacking Terpadu 6-Model: {rmse_stacked_val:.5f}")


# Bab 14: Final Prediction & Submission
Pelatihan ulang penuh (final fit) pada 100% data latih (1.054.200 baris) dengan bobot sampel temporal musim dingin dan 3 random seeds independen (42, 100, 2024). Inferensi data uji dipadukan menggunakan meta-learner terkalibrasi, dibatasi batas fisik operasional [0.02, 0.98], dipertahankan dalam format kontinu float penuh (tanpa pembulatan 3 desimal untuk mengeliminasi variansi kuantisasi), dan disinkronkan presisi dengan urutan ID resmi.


In [ ]:
print("Melatih Ulang Seluruh 6 Model pada 100% Data Latih Penuh...")

final_lgb_A_params = {k: v for k, v in lgb_A_params.items() if k != 'n_estimators'}
lgb_A_test_preds = []
t0 = time.time()
for s in SEEDS:
    print(f"Melatih LightGBM Deep Seed {s} pada 100% data ({max(100, optimal_lgb_A_iter)} pohon)...")
    m = lgb.LGBMRegressor(**final_lgb_A_params, n_estimators=max(100, optimal_lgb_A_iter), random_state=s)
    m.fit(X_train_all, y_train_all, sample_weight=weights_full)
    lgb_A_test_preds.append(np.clip(m.predict(X_test_all), 0.02, 0.98))
pred_lgb_A_test = np.mean(lgb_A_test_preds, axis=0)

X_tr_cb_all = X_train_all.copy()
X_te_cb_all = X_test_all.copy()
for cat in CATEGORICAL_COLS:
    X_tr_cb_all[cat] = X_tr_cb_all[cat].astype(str)
    X_te_cb_all[cat] = X_te_cb_all[cat].astype(str)

final_cb_A_params = {k: v for k, v in cb_A_params.items() if k != 'iterations'}
cb_A_test_preds = []
for s in SEEDS:
    print(f"Melatih CatBoost Deep Seed {s} pada 100% data ({max(100, optimal_cb_A_iter)} pohon)...")
    m = cb.CatBoostRegressor(**final_cb_A_params, iterations=max(100, optimal_cb_A_iter), random_seed=s)
    m.fit(X_tr_cb_all, y_train_all, sample_weight=weights_full, cat_features=CATEGORICAL_COLS)
    cb_A_test_preds.append(np.clip(m.predict(X_te_cb_all), 0.02, 0.98))
pred_cb_A_test = np.mean(cb_A_test_preds, axis=0)

xgb_tr_all = X_train_all.copy()
xgb_te_all = X_test_all.copy()
for c in CATEGORICAL_COLS:
    xgb_tr_all[c] = xgb_tr_all[c].cat.codes
    xgb_te_all[c] = xgb_te_all[c].cat.codes

final_xgb_A_params = {k: v for k, v in xgb_A_params.items() if k not in ['n_estimators', 'early_stopping_rounds']}
xgb_A_test_preds = []
for s in SEEDS:
    print(f"Melatih XGBoost Deep Seed {s} pada 100% data ({max(100, optimal_xgb_A_iter)} pohon)...")
    m = xgb.XGBRegressor(**final_xgb_A_params, n_estimators=max(100, optimal_xgb_A_iter), random_state=s)
    m.fit(xgb_tr_all, y_train_all, sample_weight=weights_full, verbose=False)
    xgb_A_test_preds.append(np.clip(m.predict(xgb_te_all), 0.02, 0.98))
pred_xgb_A_test = np.mean(xgb_A_test_preds, axis=0)

final_lgb_B_params = {k: v for k, v in lgb_B_params.items() if k != 'n_estimators'}
lgb_B_test_preds = []
for s in SEEDS:
    print(f"Melatih LightGBM Reg Seed {s} pada 100% data ({max(100, optimal_lgb_B_iter)} pohon)...")
    m = lgb.LGBMRegressor(**final_lgb_B_params, n_estimators=max(100, optimal_lgb_B_iter), random_state=s)
    m.fit(X_train_all, y_train_all, sample_weight=weights_full)
    lgb_B_test_preds.append(np.clip(m.predict(X_test_all), 0.02, 0.98))
pred_lgb_B_test = np.mean(lgb_B_test_preds, axis=0)

final_cb_B_params = {k: v for k, v in cb_B_params.items() if k != 'iterations'}
cb_B_test_preds = []
for s in SEEDS:
    print(f"Melatih CatBoost Reg Seed {s} pada 100% data ({max(100, optimal_cb_B_iter)} pohon)...")
    m = cb.CatBoostRegressor(**final_cb_B_params, iterations=max(100, optimal_cb_B_iter), random_seed=s)
    m.fit(X_tr_cb_all, y_train_all, sample_weight=weights_full, cat_features=CATEGORICAL_COLS)
    cb_B_test_preds.append(np.clip(m.predict(X_te_cb_all), 0.02, 0.98))
pred_cb_B_test = np.mean(cb_B_test_preds, axis=0)

final_xgb_B_params = {k: v for k, v in xgb_B_params.items() if k not in ['n_estimators', 'early_stopping_rounds']}
xgb_B_test_preds = []
for s in SEEDS:
    print(f"Melatih XGBoost Reg Seed {s} pada 100% data ({max(100, optimal_xgb_B_iter)} pohon)...")
    m = xgb.XGBRegressor(**final_xgb_B_params, n_estimators=max(100, optimal_xgb_B_iter), random_state=s)
    m.fit(xgb_tr_all, y_train_all, sample_weight=weights_full, verbose=False)
    xgb_B_test_preds.append(np.clip(m.predict(xgb_te_all), 0.02, 0.98))
pred_xgb_B_test = np.mean(xgb_B_test_preds, axis=0)

print(f"Seluruh 6 Model Berhasil Dilatih Penuh dalam {time.time()-t0:.1f} detik.")


Penggabungan inferensi data uji menggunakan meta-learner teratur, pemotongan batas fisik operasional [0.02, 0.98], preservasi presisi kontinu penuh (tanpa pembulatan kuantisasi 3 desimal), sinkronisasi ID otomatis, dan penyimpanan berkas submission_4.csv.


In [ ]:
S_test_matrix = np.column_stack([
    pred_lgb_A_test, pred_cb_A_test, pred_xgb_A_test,
    pred_lgb_B_test, pred_cb_B_test, pred_xgb_B_test
])

final_submission_preds = np.clip(meta_learner.predict(S_test_matrix), 0.02, 0.98)

submission_df = pd.DataFrame({
    'id': test_feat['id'],
    'utilization_rate': final_submission_preds
})
submission_df = test_raw[['id']].merge(submission_df, on='id', how='left')

assert len(submission_df) == len(test_raw), f"Panjang baris submission tidak cocok: {len(submission_df)} vs {len(test_raw)}"
assert not submission_df['utilization_rate'].isnull().any(), "Ditemukan nilai NaN pada berkas submission."
assert (submission_df['utilization_rate'] >= 0.02).all() and (submission_df['utilization_rate'] <= 0.98).all(), "Nilai melampaui batasan fisik stasiun."

SUBMISSION_FILENAME = 'submission_4.csv'
submission_df.to_csv(SUBMISSION_FILENAME, index=False)

if os.path.exists('submission'):
    submission_df.to_csv(os.path.join('submission', SUBMISSION_FILENAME), index=False)
elif os.path.exists('../submission'):
    submission_df.to_csv(os.path.join('../submission', SUBMISSION_FILENAME), index=False)

print(f"Berkas submission resmi berhasil dibentuk: {SUBMISSION_FILENAME}")
print(f"Dimensi berkas : {submission_df.shape[0]:,} baris x {submission_df.shape[1]} kolom")
print(f"Sebaran Statistik Prediksi Final (Presisi Kontinu Penuh):")
print(f"  Nilai Terendah : {final_submission_preds.min():.5f}")
print(f"  Nilai Tertinggi: {final_submission_preds.max():.5f}")
print(f"  Rata-rata      : {final_submission_preds.mean():.5f}")
print(f"  Standar Deviasi: {final_submission_preds.std():.5f}")
print("Sampel 10 Baris Pertama Prediksi:")
print(submission_df.head(10))


# Bab 15: Kesimpulan & Next Steps
### 15.1 Kesimpulan Analitis Eksperimen 4
1. Arsitektur Dual-Philosophy 6-Model Stacking (Aliran Deep Capacity + Aliran Conservative Regularized) menghasilkan sinergi ortogonal kuat antara pemetaan non-linear mendalam dan kontrol variansi stabilitas pohon.
2. Pembobotan sampel temporal terarah (1.15x - 1.25x pada bulan November dan jam sibuk siang hari) secara efektif memfokuskan proses optimasi model pada wilayah penyumbang galat terbesar (peak hours dan highway corridor).
3. Peniadaan pembulatan kuantisasi tiga angka desimal mempertahankan nilai ekspektasi kontinu murni E[Y|X], yang secara matematis dan empiris mereduksi variansi galat kuadratik pada metrik evaluasi RMSE.
4. Jaminan sinkronisasi ID berbasis merge menjamin keutuhan urutan baris submission secara sempurna.

### 15.2 Rencana Langkah Lanjutan (Next Steps)
1. Mengeksekusi berkas submission_4.csv pada Kaggle Leaderboard untuk memanfaatkan kuota submit ke-2 hari ini dan memverifikasi lompatan peringkat menuju Top 1.
2. Menganalisis respon Leaderboard publik terhadap penghapusan pembulatan desimal dan pembobotan sampel temporal.
3. Menyiapkan integrasi stacking akhir jika diperlukan pada slot submit berikutnya.
